# Gemma 4 Developer Agent — first slice (baseline)

Turn Gemma 4 E4B into an offline coding agent that fixes real Python bugs.
The agent loop: read the buggy file, plan a fix, edit it, run the tests,
retry up to 3 rounds, record pass/fail per bug. Each round tries up to 3
candidate fixes — a greedy (deterministic) one first, then up to 2 sampled
ones — and keeps the first that passes. That kills the run-to-run sampling
variance that flipped run 1 (5/5) to run 2 (1/10) with identical settings.

**Model:** `google/gemma-4-e4b-it-qat-w4a16-ct` — the QAT 4-bit E4B
checkpoint. Only the QAT variants are permitted submission models, so the
bf16 checkpoint is NOT used. NOTE: the kagglehub handle is
`google/gemma-4/Other/gemma-4-e4b-it-qat-w4a16-ct` (the "Other"
framework, confirmed 26 Nov from the Kaggle model page — the
"transformers" handle 404s). The checkpoint is the full multimodal
Gemma4ForConditionalGeneration model, so the load uses
`attn_implementation="eager"` — its audio sub-model does not support
SDPA (that was the 23:21 ValueError).

**Before running:**
0. If you already ran an earlier version of this notebook in this
   session, **restart the kernel first** (Session → Restart, or Runtime →
   Restart) — the old model otherwise stays in GPU memory and the new
   load runs out of memory.
1. Notebook Settings → Accelerator → **GPU L4 x4 (required)** — the
   competition's own hardware, 96 GB. The whole model fits on ONE 24 GB
   L4, so no GPU split is needed. (T4 x2 no longer works: the split
   kept hitting accelerate edge cases and the model cannot run split.)
   Attach the model as an input first: right panel →
   Input → Add input → Models → search "gemma-4" → pick the
   `e4b-it-qat-w4a16-ct` variant. The notebook loads the attached model
   first; without it, L4 has no internet so the kagglehub/HF download
   cannot work.
2. Accept the Gemma license: on Kaggle go to the `google/gemma-4` model page
   and accept, or add your Hugging Face token as a Kaggle secret named
   `HF_TOKEN` (the notebook falls back to it if kagglehub fails).
3. Run all cells top to bottom. The run takes a few minutes after the model
   loads (~2-3 min download on first run).

**What the baseline measures:** pass/fail per bug and attempts used, over
10 small real bugs (off-by-one, wrong comparison, mutable default arg,
missing dict key, wrong sort, a str/int type error, a missing import, shared
mutable state leaking between calls, a slicing off-by-one, and a function
that mutates its input). Each bug has tests that fail before the fix
and pass after. A second section runs the same loop over 11 real bugs
extracted from requests, httpx and rich — the competition's own repos —
which is the harder, more representative test. This is the starting
point — later slices add more bugs and LoRA fine-tuning.

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), "Enable GPU in Notebook Settings (L4 x4)"


In [ ]:
# Kaggle images usually have these, but install to be safe.
# compressed-tensors is needed to load the QAT w4a16 checkpoint.
%pip install -q -U transformers accelerate kagglehub compressed-tensors


In [ ]:
import sys
from pathlib import Path

# Files embedded by build_notebook.py from the repo (agent.py + bugs/).
# Written to /kaggle/working so the notebook runs from a single upload.
# ensure_ascii=False is REQUIRED here, not cosmetic: with the default
# ensure_ascii=True, astral characters (e.g. the emoji in the real_02
# super_len test) become 😀 surrogate-pair escapes in the
# cell source, which Python parses as lone surrogates — and
# Path.write_text then fails with UnicodeEncodeError on Kaggle.
# With ensure_ascii=False the literal character is embedded instead.
FILES = {
 "agent.py": "\"\"\"Agent loop for the Gemma 4 Developer Agent entry.\n\nThe loop is model-agnostic: the caller passes a ``generate`` callable that\ntakes a prompt string and a ``greedy`` flag and returns model text. The\nKaggle notebook wires that callable up to a locally loaded Gemma 4 E4B\n(QAT 4-bit) model. Nothing here makes network calls.\n\nWhy greedy first, then sampled: run 1 (5 easy bugs) was 5/5, run 2 (same\nbugs + 5 harder) was 1/10 with identical prompt, loop and generation\nsettings — pure sampling variance. So each round now tries up to 3\ncandidate fixes: a greedy (deterministic) fix first, then up to 2 sampled\nfixes (temperature 0.7). Each candidate is tested against the bug's tests\nand the first one that passes is kept. A \"round\" is one attempt in the\nresults table (max 3 rounds per bug, so up to 9 model calls per bug worst\ncase).\n\nFlow per bug:\n    1. copy the buggy file + its tests into a fresh working directory\n    2. run the tests (they should fail at first)\n    3. per round: ask the model for a fix (greedy, then sampled), test\n       each candidate, keep the first that passes\n    4. retry up to ``max_attempts`` rounds, then record pass/fail\n\"\"\"\n\nfrom __future__ import annotations\n\nimport re\nimport shutil\nimport subprocess\nimport sys\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\n\n@dataclass\nclass BugResult:\n    \"\"\"One bug, one row of the baseline table.\"\"\"\n\n    name: str\n    attempts: int = 0\n    passed: bool = False\n    log: list[str] = field(default_factory=list)\n\n\nCODE_BLOCK_RE = re.compile(r\"```(?:python)?\\n(.*?)```\", re.DOTALL)\n\n\ndef extract_code(text: str) -> str:\n    \"\"\"Pull the first python code block out of model output.\n\n    Falls back to the whole output if there is no fenced block, so a model\n    that just prints code still works.\n    \"\"\"\n    match = CODE_BLOCK_RE.search(text)\n    if match:\n        return match.group(1).strip() + \"\\n\"\n    return text.strip() + \"\\n\"\n\n\ndef build_prompt(code: str, test_code: str, test_output: str) -> str:\n    \"\"\"Prompt the model with the buggy file, its tests, and the failure.\"\"\"\n    return (\n        \"You are fixing a small Python bug. Reply with ONLY the complete \"\n        \"corrected file inside one ```python code block, nothing else.\\n\\n\"\n        \"The file is supposed to do what its docstring says. \"\n        \"The tests below describe the correct behaviour.\\n\\n\"\n        \"BUGGY FILE:\\n```python\\n\" + code + \"```\\n\\n\"\n        \"TESTS:\\n```python\\n\" + test_code + \"```\\n\\n\"\n        \"FAILING TEST OUTPUT (last 40 lines):\\n```\\n\"\n        + \"\\n\".join(test_output.strip().splitlines()[-40:])\n        + \"\\n```\\n\\n\"\n        \"CORRECTED FILE:\\n```python\\n\"\n    )\n\n\ndef run_tests(test_path: Path, cwd: Path, timeout: int = 120) -> tuple[bool, str]:\n    \"\"\"Run one test file with pytest. Returns (passed, combined output).\"\"\"\n    proc = subprocess.run(\n        [sys.executable, \"-m\", \"pytest\", \"-q\", \"-p\", \"no:cacheprovider\", str(test_path)],\n        cwd=cwd,\n        capture_output=True,\n        text=True,\n        timeout=timeout,\n    )\n    return proc.returncode == 0, proc.stdout + proc.stderr\n\n\ndef run_agent(\n    generate,\n    bug_file: Path,\n    test_file: Path,\n    work_root: Path,\n    max_attempts: int = 3,\n    candidates_per_round: int = 3,\n) -> BugResult:\n    \"\"\"Run the fix loop for one bug. Edits only a copy, never the original.\n\n    Each round tries up to ``candidates_per_round`` fixes: the first is\n    greedy (deterministic), the rest are sampled. Each candidate is tested\n    and the first one that passes wins the round. A round counts as one\n    attempt in the results table.\n    \"\"\"\n    name = bug_file.stem\n    result = BugResult(name=name)\n    workdir = work_root / name\n    testdir = workdir / \"tests\"\n    testdir.mkdir(parents=True, exist_ok=True)\n\n    work_bug = workdir / bug_file.name\n    work_test = testdir / test_file.name\n    shutil.copy(bug_file, work_bug)\n    shutil.copy(test_file, work_test)\n    # conftest.py makes the bug module importable from the tests dir.\n    conftest = test_file.parent / \"conftest.py\"\n    if conftest.exists():\n        shutil.copy(conftest, testdir / \"conftest.py\")\n\n    code = work_bug.read_text()\n    test_code = work_test.read_text()\n\n    passed, output = run_tests(work_test, cwd=workdir)\n    result.log.append(\"attempt 0 (original): \" + (\"PASS\" if passed else \"FAIL\"))\n    if passed:\n        result.passed = True\n        return result\n\n    for attempt in range(1, max_attempts + 1):\n        result.attempts = attempt\n        prompt = build_prompt(code, test_code, output)\n        last_fixed = None\n        for cand in range(candidates_per_round):\n            greedy = cand == 0\n            mode = \"greedy\" if greedy else \"sampled\"\n            try:\n                fixed = extract_code(generate(prompt, greedy=greedy))\n            except Exception as exc:  # model call failed, try next candidate\n                result.log.append(\n                    f\"attempt {attempt} candidate {cand + 1} ({mode}): \"\n                    f\"generate() raised {exc!r}\"\n                )\n                continue\n            last_fixed = fixed\n            work_bug.write_text(fixed)\n            passed, output = run_tests(work_test, cwd=workdir)\n            result.log.append(\n                f\"attempt {attempt} candidate {cand + 1} ({mode}): \"\n                + (\"PASS\" if passed else \"FAIL\")\n            )\n            if passed:\n                result.passed = True\n                return result\n        # Next round refines from the last candidate tried, as before.\n        if last_fixed is not None:\n            code = last_fixed\n        work_bug.write_text(code)\n\n    return result\n\n\ndef run_all(\n    generate,\n    bugs_dir: Path,\n    work_root: Path,\n    max_attempts: int = 3,\n    candidates_per_round: int = 3,\n    pattern: str = \"bug_*.py\",\n) -> list[BugResult]:\n    \"\"\"Run the agent over every file matching ``pattern`` in bugs_dir (sorted by name).\n\n    Default pattern is the synthetic dev set (``bug_*.py``); pass\n    ``pattern=\"real_*.py\"`` for the real-bug set in ``real_bugs/``.\n    \"\"\"\n    results = []\n    for bug_file in sorted(bugs_dir.glob(pattern)):\n        test_file = bugs_dir / \"tests\" / f\"test_{bug_file.stem}.py\"\n        if not test_file.exists():\n            print(f\"SKIP {bug_file.name}: no {test_file.name}\")\n            continue\n        print(f\"--- {bug_file.name} ---\")\n        res = run_agent(\n            generate, bug_file, test_file, work_root, max_attempts, candidates_per_round\n        )\n        print(f\"    {'PASS' if res.passed else 'FAIL'} after {res.attempts} attempt(s)\")\n        results.append(res)\n    return results\n\n\ndef results_table(results: list[BugResult]) -> str:\n    \"\"\"Markdown table for baseline_results.md.\"\"\"\n    lines = [\n        \"| Bug | Attempts | Result |\",\n        \"| --- | --- | --- |\",\n    ]\n    for res in results:\n        lines.append(f\"| {res.name} | {res.attempts} | {'pass' if res.passed else 'fail'} |\")\n    passed = sum(r.passed for r in results)\n    lines.append(\"\")\n    lines.append(f\"Pass rate: {passed}/{len(results)}\")\n    return \"\\n\".join(lines) + \"\\n\"\n",
 "bugs/bug_01_off_by_one.py": "\"\"\"Add up a list of numbers.\n\nBug: the total is wrong, it misses the last number.\n\"\"\"\n\n\ndef total(xs):\n    s = 0\n    for i in range(len(xs) - 1):\n        s += xs[i]\n    return s\n",
 "bugs/bug_02_wrong_comparison.py": "\"\"\"Check whether someone is an adult (18 or older).\"\"\"\n\n\ndef is_adult(age):\n    return age > 18\n",
 "bugs/bug_03_mutable_default.py": "\"\"\"Add an item to a shopping cart and return the cart.\n\nEach call without a cart should start from an empty cart.\n\"\"\"\n\n\ndef add_item(item, cart=[]):\n    cart.append(item)\n    return cart\n",
 "bugs/bug_04_missing_key.py": "\"\"\"Look up the price of an item.\n\nUnknown items cost 0 instead of crashing.\n\"\"\"\n\nPRICES = {\"apple\": 10, \"banana\": 5, \"orange\": 8}\n\n\ndef get_price(item):\n    return PRICES[item]\n",
 "bugs/bug_05_wrong_sort.py": "\"\"\"Return the 3 highest scores, highest first.\"\"\"\n\n\ndef top_scores(scores):\n    return sorted(scores)[:3]\n",
 "bugs/bug_06_type_error.py": "\"\"\"Apply a discount to a price loaded from a config dict.\n\nPrices in the config are stored as strings, so they must be converted to\nnumbers before doing arithmetic.\n\"\"\"\n\nCONFIG = {\"price\": \"120\", \"discount_pct\": 10}\n\n\ndef final_price():\n    return CONFIG[\"price\"] * (1 - CONFIG[\"discount_pct\"] / 100)\n",
 "bugs/bug_07_missing_import.py": "\"\"\"Load a JSON string and return the value for a key.\"\"\"\n\n\ndef get_setting(payload, key):\n    data = json.loads(payload)\n    return data[key]\n",
 "bugs/bug_08_shared_state.py": "\"\"\"Collect the even numbers from a list.\n\nReturn only the even numbers from the list given in this call.\n\"\"\"\n\n_seen = []\n\n\ndef evens(xs):\n    for x in xs:\n        if x % 2 == 0:\n            _seen.append(x)\n    return _seen\n",
 "bugs/bug_09_slice_off_by_one.py": "\"\"\"Return page `page` (1-based) of `size` items from a list.\"\"\"\n\n\ndef page(items, page, size):\n    start = (page - 1) * size\n    return items[start:start + size - 1]\n",
 "bugs/bug_10_mutates_input.py": "\"\"\"Return a new list with a bonus added to each score.\n\nThe original list must not be changed.\n\"\"\"\n\n\ndef with_bonus(scores, bonus):\n    out = scores\n    for i in range(len(out)):\n        out[i] += bonus\n    return out\n",
 "bugs/tests/conftest.py": "\"\"\"Makes the parent bugs/ directory importable so tests can import the bug modules.\"\"\"\nimport os\nimport sys\n\nsys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), \"..\")))\n",
 "bugs/tests/test_bug_01_off_by_one.py": "from bug_01_off_by_one import total\n\n\ndef test_total_all_numbers():\n    assert total([1, 2, 3]) == 6\n\n\ndef test_total_single():\n    assert total([7]) == 7\n\n\ndef test_total_empty():\n    assert total([]) == 0\n",
 "bugs/tests/test_bug_02_wrong_comparison.py": "from bug_02_wrong_comparison import is_adult\n\n\ndef test_eighteen_is_adult():\n    assert is_adult(18) is True\n\n\ndef test_twenty_is_adult():\n    assert is_adult(20) is True\n\n\ndef test_seventeen_is_not_adult():\n    assert is_adult(17) is False\n",
 "bugs/tests/test_bug_03_mutable_default.py": "from bug_03_mutable_default import add_item\n\n\ndef test_first_call():\n    assert add_item(\"apple\") == [\"apple\"]\n\n\ndef test_carts_are_independent():\n    a = add_item(\"apple\")\n    b = add_item(\"banana\")\n    assert a == [\"apple\"]\n    assert b == [\"banana\"]\n\n\ndef test_passed_cart_is_used():\n    assert add_item(\"pear\", [\"plum\"]) == [\"plum\", \"pear\"]\n",
 "bugs/tests/test_bug_04_missing_key.py": "from bug_04_missing_key import get_price\n\n\ndef test_known_item():\n    assert get_price(\"apple\") == 10\n\n\ndef test_unknown_item_is_zero():\n    assert get_price(\"mango\") == 0\n\n\ndef test_another_unknown_item():\n    assert get_price(\"kiwi\") == 0\n",
 "bugs/tests/test_bug_05_wrong_sort.py": "from bug_05_wrong_sort import top_scores\n\n\ndef test_top_three_highest_first():\n    assert top_scores([5, 1, 9, 3]) == [9, 5, 3]\n\n\ndef test_fewer_than_three():\n    assert top_scores([2, 7]) == [7, 2]\n\n\ndef test_already_sorted():\n    assert top_scores([10, 8, 6, 4]) == [10, 8, 6]\n",
 "bugs/tests/test_bug_06_type_error.py": "import bug_06_type_error as mod\n\n\ndef test_default_config():\n    mod.CONFIG = {\"price\": \"120\", \"discount_pct\": 10}\n    assert mod.final_price() == 108.0\n\n\ndef test_quarter_off():\n    mod.CONFIG = {\"price\": \"200\", \"discount_pct\": 25}\n    assert mod.final_price() == 150.0\n\n\ndef test_no_discount():\n    mod.CONFIG = {\"price\": \"50\", \"discount_pct\": 0}\n    assert mod.final_price() == 50.0\n",
 "bugs/tests/test_bug_07_missing_import.py": "from bug_07_missing_import import get_setting\n\n\ndef test_string_value():\n    assert get_setting('{\"theme\": \"dark\"}', \"theme\") == \"dark\"\n\n\ndef test_int_value():\n    assert get_setting('{\"retries\": 3}', \"retries\") == 3\n\n\ndef test_second_key():\n    assert get_setting('{\"a\": 1, \"b\": 2}', \"b\") == 2\n",
 "bugs/tests/test_bug_08_shared_state.py": "from bug_08_shared_state import evens\n\n\ndef test_evens_first_call():\n    assert evens([1, 2, 3, 4]) == [2, 4]\n    # A second identical call must give the same answer, not accumulate.\n    assert evens([1, 2, 3, 4]) == [2, 4]\n\n\ndef test_evens_second_call_is_fresh():\n    evens([1, 2, 3, 4])\n    assert evens([5, 6]) == [6]\n\n\ndef test_evens_empty_after_calls():\n    evens([2, 4])\n    assert evens([]) == []\n",
 "bugs/tests/test_bug_09_slice_off_by_one.py": "from bug_09_slice_off_by_one import page\n\n\ndef test_first_page():\n    assert page([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 1, 3) == [1, 2, 3]\n\n\ndef test_second_page():\n    assert page([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 2, 3) == [4, 5, 6]\n\n\ndef test_second_page_size_two():\n    assert page([1, 2, 3, 4, 5], 2, 2) == [3, 4]\n",
 "bugs/tests/test_bug_10_mutates_input.py": "from bug_10_mutates_input import with_bonus\n\n\ndef test_bonus_added_and_input_unchanged():\n    scores = [10, 20]\n    assert with_bonus(scores, 5) == [15, 25]\n    assert scores == [10, 20]\n\n\ndef test_bonus_added_larger_list():\n    scores = [1, 2, 3]\n    assert with_bonus(scores, 10) == [11, 12, 13]\n    assert scores == [1, 2, 3]\n\n\ndef test_repeated_calls_same_input():\n    scores = [4]\n    first = with_bonus(scores, 1)\n    second = with_bonus(scores, 1)\n    assert first == [5]\n    assert second == [5]\n    assert scores == [4]\n",
 "real_bugs/real_01_requests_content_type.py": "\"\"\"Parse a Content-Type header into (content_type, params).\n\nExtracted from requests, src/requests/utils.py, as it was before\nPR #7309 (\"Fix malformed value parsing for Content-Type\",\nNate Prewitt, 2026-03-30).\n\nThe bug: a parameter with no ``=`` in it (a malformed value such as\n``; no_equals``) is stored in the params dict with the value ``True``\ninstead of being ignored.\n\"\"\"\n\n\ndef _parse_content_type_header(header):\n    \"\"\"Returns content type and parameters from given header\n\n    :param header: string\n    :return: tuple containing content type and dictionary of\n        parameters\n    \"\"\"\n\n    tokens = header.split(\";\")\n    content_type, params = tokens[0].strip(), tokens[1:]\n    params_dict = {}\n    items_to_strip = \"\\\"' \"\n\n    for param in params:\n        param = param.strip()\n        if param:\n            key, value = param, True\n            index_of_equals = param.find(\"=\")\n            if index_of_equals != -1:\n                key = param[:index_of_equals].strip(items_to_strip)\n                value = param[index_of_equals + 1 :].strip(items_to_strip)\n            params_dict[key.lower()] = value\n    return content_type, params_dict\n",
 "real_bugs/real_02_requests_super_len.py": "\"\"\"Return the length of an object in bytes.\n\nExtracted from requests, src/requests/utils.py, as it was before\ncommit 3fd309a (\"Enhance `super_len` to count encoded bytes for str\",\nBruce Adams, 2023-11-28, fixing issue #6586).\n\nThe bug: a ``str`` is measured in characters, not in UTF-8 bytes, so\n``super_len(\"caf\\u00e9\")`` returns 4 instead of 5 and the Content-Length\nsent for a non-ASCII string body is too small.\n\"\"\"\n\nimport io\nimport os\nimport warnings\n\n\nclass FileModeWarning(Warning):\n    \"\"\"A file was opened in text mode, but Requests determined its\n    binary size.\"\"\"\n\n\ndef super_len(o):\n    total_length = None\n    current_position = 0\n\n    if hasattr(o, \"__len__\"):\n        total_length = len(o)\n\n    elif hasattr(o, \"len\"):\n        total_length = o.len\n\n    elif hasattr(o, \"fileno\"):\n        try:\n            fileno = o.fileno()\n        except (io.UnsupportedOperation, AttributeError):\n            # AttributeError is a surprising exception, seeing as how we've just checked\n            # that `hasattr(o, 'fileno')`.  It happens for objects obtained via\n            # `Tarfile.extractfile()`, per issue 5229.\n            pass\n        else:\n            total_length = os.fstat(fileno).st_size\n\n            # Having used fstat to determine the file length, we need to\n            # confirm that this file was opened up in binary mode.\n            if \"b\" not in o.mode:\n                warnings.warn(\n                    (\n                        \"Requests has determined the content-length for this \"\n                        \"request using the binary size of the file: however, the \"\n                        \"file has been opened in text mode (i.e. without the 'b' \"\n                        \"flag in the mode). This may lead to an incorrect \"\n                        \"content-length. In Requests 3.0, support will be removed \"\n                        \"for files in text mode.\"\n                    ),\n                    FileModeWarning,\n                )\n\n    if hasattr(o, \"tell\"):\n        try:\n            current_position = o.tell()\n        except OSError:\n            # This can happen in some weird situations, such as when the file\n            # is actually a special file descriptor like stdin. In this\n            # instance, we don't know what the length is, so set it to zero and\n            # let requests chunk it instead.\n            if total_length is not None:\n                current_position = total_length\n        else:\n            if hasattr(o, \"seek\") and total_length is None:\n                # StringIO and BytesIO have seek but no usable fileno\n                try:\n                    # seek to end of file\n                    o.seek(0, 2)\n                    total_length = o.tell()\n\n                    # seek back to current position to support\n                    # partially read file-like objects\n                    o.seek(current_position or 0)\n                except OSError:\n                    total_length = 0\n\n    if total_length is None:\n        total_length = 0\n\n    return max(0, total_length - current_position)\n",
 "real_bugs/real_03_requests_netrc_auth.py": "\"\"\"Return the Requests tuple auth for a given url from netrc.\n\nExtracted from requests, src/requests/utils.py, as it was before\nPR #7205 (\"Fix empty netrc entry usage\", Nate Prewitt, 2026-02-12).\n\nThe bug: an empty netrc entry (e.g. a bare ``default`` line, which\nparses to ``(None, None, None)``) is truthy as a tuple, so\n``get_netrc_auth`` returns ``(None, None)`` instead of ``None``.\n\"\"\"\n\nimport os\nfrom urllib.parse import urlparse\n\nNETRC_FILES = (\".netrc\", \"_netrc\")\n\n\ndef get_netrc_auth(url, raise_errors=False):\n    \"\"\"Returns the Requests tuple auth for a given url from netrc.\"\"\"\n\n    netrc_file = os.environ.get(\"NETRC\")\n    if netrc_file is not None:\n        netrc_locations = (netrc_file,)\n    else:\n        netrc_locations = (f\"~/{f}\" for f in NETRC_FILES)\n\n    try:\n        from netrc import NetrcParseError, netrc\n\n        netrc_path = None\n\n        for f in netrc_locations:\n            loc = os.path.expanduser(f)\n            if os.path.exists(loc):\n                netrc_path = loc\n                break\n\n        # Abort early if there isn't one.\n        if netrc_path is None:\n            return\n\n        ri = urlparse(url)\n        host = ri.hostname\n\n        try:\n            _netrc = netrc(netrc_path).authenticators(host)\n            if _netrc:\n                # Return with login / password\n                login_i = 0 if _netrc[0] else 1\n                return (_netrc[login_i], _netrc[2])\n        except (NetrcParseError, OSError):\n            # If there was a parsing error or a permissions issue reading the file,\n            # we'll just skip netrc auth unless explicitly asked to raise errors.\n            if raise_errors:\n                raise\n\n    # App Engine hackiness.\n    except (ImportError, AttributeError):\n        pass\n",
 "real_bugs/real_04_httpx_primitive_value.py": "\"\"\"Coerce a primitive data type into a string value.\n\nExtracted from httpx, httpx/_utils.py, as it was before PR #2523\n(\"Raise `TypeError` on invalid query params\", Tom Christie,\n2022-12-30).\n\nThe bug: any value at all is silently stringified, so\n``QueryParams({\"a\": b\"bytes\"})`` produces ``a=b'bytes'`` instead of\nraising a ``TypeError``.\n\"\"\"\n\n\ndef primitive_value_to_str(value):\n    \"\"\"\n    Coerce a primitive data type into a string value.\n\n    Note that we prefer JSON-style 'true'/'false' for boolean values here.\n    \"\"\"\n    if value is True:\n        return \"true\"\n    elif value is False:\n        return \"false\"\n    elif value is None:\n        return \"\"\n    return str(value)\n",
 "real_bugs/real_05_httpx_text_chunker.py": "\"\"\"Split decoded text content into fixed-size chunks.\n\nExtracted from httpx, httpx/_decoders.py, as it was before commit\n1e11096 (\"Fixed `iter_text` adding an empty string\", James Braza,\n2023-12-11, fixing issue #2998).\n\nThe bug: ``decode(\"\")`` returns ``[\"\"]`` instead of ``[]``, so\niterating the text of an empty response yields one empty string.\n\"\"\"\n\nimport io\nimport typing\n\n\nclass TextChunker:\n    \"\"\"\n    Handles returning text content in fixed-size chunks.\n    \"\"\"\n\n    def __init__(self, chunk_size: typing.Optional[int] = None) -> None:\n        self._buffer = io.StringIO()\n        self._chunk_size = chunk_size\n\n    def decode(self, content: str) -> typing.List[str]:\n        if self._chunk_size is None:\n            return [content]\n\n        self._buffer.write(content)\n        if self._buffer.tell() >= self._chunk_size:\n            value = self._buffer.getvalue()\n            chunks = [\n                value[i : i + self._chunk_size]\n                for i in range(0, len(value), self._chunk_size)\n            ]\n            if len(chunks[-1]) == self._chunk_size:\n                self._buffer.seek(0)\n                self._buffer.truncate()\n                return chunks\n            else:\n                self._buffer.seek(0)\n                self._buffer.write(chunks[-1])\n                self._buffer.truncate()\n                return chunks[:-1]\n        else:\n            return []\n\n    def flush(self) -> typing.List[str]:\n        value = self._buffer.getvalue()\n        self._buffer.seek(0)\n        self._buffer.truncate()\n        return [value] if value else []\n",
 "real_bugs/real_06_httpx_no_proxy.py": "\"\"\"Get proxy information from the environment.\n\nExtracted from httpx, httpx/_utils.py, as it was before PR #2659\n(\"fix: NO_PROXY should support IPv4, IPv6 and localhost\",\nJiayun Shen, 2023-04-19).\n\nThe bug: every NO_PROXY entry is treated as a domain suffix, so\n``NO_PROXY=127.0.0.1`` becomes the mount ``all://*127.0.0.1`` instead\nof ``all://127.0.0.1``; IPv6 addresses and ``localhost`` are wrong\nthe same way.\n\"\"\"\n\nimport os\nimport typing\nfrom urllib.request import getproxies\n\n\ndef get_environment_proxies() -> typing.Dict[str, typing.Optional[str]]:\n    \"\"\"Gets proxy information from the environment\"\"\"\n\n    # urllib.request.getproxies() falls back on System\n    # Registry and Config for proxies on Windows and macOS.\n    # We don't want to propagate non-HTTP proxies into\n    # our configuration such as 'TRAVIS_APT_PROXY'.\n    proxy_info = getproxies()\n    mounts: typing.Dict[str, typing.Optional[str]] = {}\n\n    for scheme in (\"http\", \"https\", \"all\"):\n        if proxy_info.get(scheme):\n            hostname = proxy_info[scheme]\n            mounts[f\"{scheme}://\"] = (\n                hostname if \"://\" in hostname else f\"http://{hostname}\"\n            )\n\n    no_proxy_hosts = [host.strip() for host in proxy_info.get(\"no\", \"\").split(\",\")]\n    for hostname in no_proxy_hosts:\n        # See https://curl.haxx.se/libcurl/c/CURLOPT_NOPROXY.html for details\n        # on how names in `NO_PROXY` are handled.\n        if hostname == \"*\":\n            # If NO_PROXY=* is used or if \"*\" occurs as any one of the comma\n            # separated hostnames, then we should just bypass any information\n            # from HTTP_PROXY, HTTPS_PROXY, ALL_PROXY, and always ignore\n            # proxies.\n            return {}\n        elif hostname:\n            # NO_PROXY=.google.com is marked as \"all://*.google.com,\n            #   which disables \"www.google.com\" but not \"google.com\"\n            # NO_PROXY=google.com is marked as \"all://*google.com,\n            #   which disables \"www.google.com\" and \"google.com\".\n            #   (But not \"wwwgoogle.com\")\n            mounts[f\"all://*{hostname}\"] = None\n\n    return mounts\n",
 "real_bugs/real_07_httpx_urlencode_space.py": "\"\"\"Build a query string from (key, value) pairs.\n\nThe ``quote`` / ``percent_encode`` helpers are extracted from httpx,\nhttpx/_urlparse.py. ``build_query_string`` is the behaviour of\n``QueryParams.__str__`` in httpx/_urls.py as it was before PR #2543\n(\"Use '%20' for encoding spaces in query parameters\", Tom Christie,\n2023-01-10, fixing issue #2536).\n\nThe bug: the standard library ``urlencode`` is used, which encodes\nspaces as ``+`` and ``/`` as ``%2F``. httpx query parameters should\nuse the same escaping as the rest of the URL: spaces as ``%20`` and\n``/`` left alone.\n\"\"\"\n\nimport re\nimport typing\nfrom urllib.parse import urlencode as _stdlib_urlencode\n\n# https://datatracker.ietf.org/doc/html/rfc3986.html#section-2.3\nUNRESERVED_CHARACTERS = (\n    \"ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789-._~\"\n)\nSUB_DELIMS = \"!$&'()*+,;=\"\n\nPERCENT_ENCODED_REGEX = re.compile(\"%[A-Fa-f0-9]{2}\")\n\n\ndef percent_encode(char: str) -> str:\n    \"\"\"\n    Replace every character in a string with the percent-encoded representation.\n\n    Characters outside the ASCII range are represented with their a percent-encoded\n    representation of their UTF-8 byte sequence.\n\n    For example:\n\n        percent_encode(\" \") == \"%20\"\n    \"\"\"\n    return \"\".join([f\"%{byte:02x}\" for byte in char.encode(\"utf-8\")]).upper()\n\n\ndef quote(string: str, safe: str = \"/\") -> str:\n    NON_ESCAPED_CHARS = UNRESERVED_CHARACTERS + safe\n    if string.count(\"%\") == len(PERCENT_ENCODED_REGEX.findall(string)):\n        # If all occurances of '%' are valid '%xx' escapes, then treat\n        # percent as a non-escaping character.\n        NON_ESCAPED_CHARS += \"%\"\n\n    return \"\".join(\n        [char if char in NON_ESCAPED_CHARS else percent_encode(char) for char in string]\n    )\n\n\ndef build_query_string(items: typing.List[typing.Tuple[str, str]]) -> str:\n    return _stdlib_urlencode(items)\n",
 "real_bugs/real_08_httpx_ipv4_regex.py": "\"\"\"Encode a hostname for use in a URL.\n\nExtracted from httpx, httpx/_urlparse.py, as it was before PR #2886\n(\"Fix encode host\", robinaly, 2023-10-10).\n\nThe bug: the IPv4-detection regex uses unescaped ``.`` which matches\nany character, so a hostname like ``023b76x43144`` is treated as an\nIPv4 address and rejected as invalid.\n\nNote: the IDNA branch below is the real code; ``idna`` is imported\nlazily here so this module stays importable without the package\ninstalled. The tested paths (IPv4-style, valid IPv4, ASCII hostname)\nare identical to the real code.\n\"\"\"\n\nimport ipaddress\nimport re\nimport typing\n\ntry:\n    import idna\nexcept ImportError:  # pragma: no cover - the real code imports idna at module level\n    idna = None\n\n# https://datatracker.ietf.org/doc/html/rfc3986.html#section-2.3\nUNRESERVED_CHARACTERS = (\n    \"ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789-._~\"\n)\nSUB_DELIMS = \"!$&'()*+,;=\"\n\nPERCENT_ENCODED_REGEX = re.compile(\"%[A-Fa-f0-9]{2}\")\n\n# We use these simple regexs as a first pass before handing off to\n# the stdlib 'ipaddress' module for IP address validation.\nIPv4_STYLE_HOSTNAME = re.compile(r\"^[0-9]+.[0-9]+.[0-9]+.[0-9]+$\")\nIPv6_STYLE_HOSTNAME = re.compile(r\"^\\[.*\\]$\")\n\n\nclass InvalidURL(Exception):\n    pass\n\n\ndef percent_encode(char: str) -> str:\n    return \"\".join([f\"%{byte:02x}\" for byte in char.encode(\"utf-8\")]).upper()\n\n\ndef quote(string: str, safe: str = \"/\") -> str:\n    NON_ESCAPED_CHARS = UNRESERVED_CHARACTERS + safe\n    if string.count(\"%\") == len(PERCENT_ENCODED_REGEX.findall(string)):\n        NON_ESCAPED_CHARS += \"%\"\n\n    return \"\".join(\n        [char if char in NON_ESCAPED_CHARS else percent_encode(char) for char in string]\n    )\n\n\ndef encode_host(host: str) -> str:\n    if not host:\n        return \"\"\n\n    elif IPv4_STYLE_HOSTNAME.match(host):\n        # Validate IPv4 hostnames like #.#.#.#\n        #\n        # From https://datatracker.ietf.org/doc/html/rfc3986/#section-3.2.2\n        #\n        # IPv4address = dec-octet \".\" dec-octet \".\" dec-octet \".\" dec-octet\n        try:\n            ipaddress.IPv4Address(host)\n        except ipaddress.AddressValueError:\n            raise InvalidURL(f\"Invalid IPv4 address: {host!r}\")\n        return host\n\n    elif IPv6_STYLE_HOSTNAME.match(host):\n        # Validate IPv6 hostnames like [...]\n        #\n        # From https://datatracker.ietf.org/doc/html/rfc3986/#section-3.2.2\n        #\n        # \"A host identified by an Internet Protocol literal address, version 6\n        # [RFC3513] or later, is distinguished by enclosing the IP literal\n        # within square brackets (\"[\" and \"]\").  This is the only place where\n        # square bracket characters are allowed in the URI syntax.\"\n        try:\n            ipaddress.IPv6Address(host[1:-1])\n        except ipaddress.AddressValueError:\n            raise InvalidURL(f\"Invalid IPv6 address: {host!r}\")\n        return host[1:-1]\n\n    elif host.isascii():\n        # Regular ASCII hostnames\n        #\n        # From https://datatracker.ietf.org/doc/html/rfc3986/#section-3.2.2\n        #\n        # reg-name    = *( unreserved / pct-encoded / sub-delims )\n        return quote(host.lower(), safe=SUB_DELIMS)\n\n    # IDNA hostnames\n    if idna is None:\n        raise InvalidURL(f\"Invalid IDNA hostname: {host!r}\")\n    try:\n        return idna.encode(host.lower()).decode(\"ascii\")\n    except idna.IDNAError:\n        raise InvalidURL(f\"Invalid IDNA hostname: {host!r}\")\n",
 "real_bugs/real_09_httpx_gen_delims.py": "\"\"\"Percent-encode the path, query and fragment of a URL.\n\nThe ``quote`` / ``percent_encode`` helpers are extracted from httpx,\nhttpx/_urlparse.py. ``encode_url_parts`` is the quoting step of\n``urlparse`` in the same file, as it was before PR #2701 (\"Fix for\ngen-delims escaping behaviour in path/query/fragment\", Tom Christie,\n2023-05-09).\n\nThe bug: the ``safe`` character sets passed to ``quote`` are wrong\nfor each component. ``:`` ``[`` ``]`` ``@`` are dropped from the path\nset, ``:`` ``[`` ``]`` ``@`` from the query set, and ``:`` ``#``\n``[`` ``]`` ``@`` from the fragment set, so characters that are legal\ndelimiters in that component get percent-encoded.\n\"\"\"\n\nimport re\nimport typing\n\n# https://datatracker.ietf.org/doc/html/rfc3986.html#section-2.3\nUNRESERVED_CHARACTERS = (\n    \"ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789-._~\"\n)\nSUB_DELIMS = \"!$&'()*+,;=\"\n\nPERCENT_ENCODED_REGEX = re.compile(\"%[A-Fa-f0-9]{2}\")\n\n\ndef percent_encode(char: str) -> str:\n    \"\"\"\n    Replace every character in a string with the percent-encoded representation.\n\n    Characters outside the ASCII range are represented with their a percent-encoded\n    representation of their UTF-8 byte sequence.\n\n    For example:\n\n        percent_encode(\" \") == \"%20\"\n    \"\"\"\n    return \"\".join([f\"%{byte:02x}\" for byte in char.encode(\"utf-8\")]).upper()\n\n\ndef quote(string: str, safe: str = \"/\") -> str:\n    NON_ESCAPED_CHARS = UNRESERVED_CHARACTERS + safe\n    if string.count(\"%\") == len(PERCENT_ENCODED_REGEX.findall(string)):\n        # If all occurances of '%' are valid '%xx' escapes, then treat\n        # percent as a non-escaping character.\n        NON_ESCAPED_CHARS += \"%\"\n\n    return \"\".join(\n        [char if char in NON_ESCAPED_CHARS else percent_encode(char) for char in string]\n    )\n\n\ndef encode_url_parts(\n    path: str,\n    query: typing.Optional[str],\n    fragment: typing.Optional[str],\n) -> typing.Tuple[str, typing.Optional[str], typing.Optional[str]]:\n    \"\"\"Percent-encode a URL's path, query and fragment components.\"\"\"\n    parsed_path: str = quote(path, safe=SUB_DELIMS + \":@/\")\n    parsed_query: typing.Optional[str] = (\n        None if query is None else quote(query, safe=SUB_DELIMS + \"/?\")\n    )\n    parsed_fragment: typing.Optional[str] = (\n        None if fragment is None else quote(fragment, safe=SUB_DELIMS + \"/?\")\n    )\n    return parsed_path, parsed_query, parsed_fragment\n",
 "real_bugs/real_10_rich_escape.py": "\"\"\"Escape text so that it won't be interpreted as markup.\n\nExtracted from rich, rich/markup.py, as it was before commit 4b1293e\n(\"fix at escape\", Will McGugan, 2022-01-02, released in 10.16.2).\n\nThe bug: the escape regex does not include ``@`` in the set of tag\nstarting characters, so ``[@click]...[/@click]`` markup is not\nescaped and is still interpreted as a tag.\n\"\"\"\n\nimport re\nfrom typing import Callable, Match\n\n_ReStringMatch = Match[str]  # regex match object\n_ReSubCallable = Callable[[_ReStringMatch], str]  # Callable invoked by re.sub\n_EscapeSubMethod = Callable[[_ReSubCallable, str], str]  # Sub method of a compiled re\n\n\ndef escape(\n    markup: str, _escape: _EscapeSubMethod = re.compile(r\"(\\\\*)(\\[[a-z#\\/].*?\\])\").sub\n) -> str:\n    \"\"\"Escapes text so that it won't be interpreted as markup.\n\n    Args:\n        markup (str): Content to be inserted in to markup.\n\n    Returns:\n        str: Markup with square brackets escaped.\n    \"\"\"\n\n    def escape_backslashes(match: Match[str]) -> str:\n        \"\"\"Called by re.sub replace matches.\"\"\"\n        backslashes, text = match.groups()\n        return f\"{backslashes}{backslashes}\\\\{text}\"\n\n    markup = _escape(escape_backslashes, markup)\n    return markup\n",
 "real_bugs/real_11_rich_markup_parse.py": "\"\"\"Parse console markup into (position, text, tag) tuples.\n\n``Tag`` and ``_parse`` are extracted from rich, rich/markup.py, as\nthey were before commit 71325bb (\"Fix issue with markup escaping\",\nDarren Burns, 2022-02-10).\n\nThe bug: the tag regex ``[a-z#\\\\/@].*?`` lets ``.*?`` span across a\n``[``, so in ``\"[foo bar [baz] qux]\"`` the whole ``\"[foo bar [baz]\"``\nis read as one tag instead of plain text ``\"[foo bar \"`` followed by\nthe tag ``[baz]``.\n\"\"\"\n\nimport re\nfrom typing import Iterable, NamedTuple, Optional, Tuple\n\nRE_TAGS = re.compile(\n    r\"\"\"((\\\\*)\\[([a-z#\\/@].*?)\\])\"\"\",\n    re.VERBOSE,\n)\n\n\nclass Tag(NamedTuple):\n    \"\"\"A tag in console markup.\"\"\"\n\n    name: str\n    \"\"\"The tag name. e.g. 'bold'.\"\"\"\n    parameters: Optional[str]\n    \"\"\"Any additional parameters after the name.\"\"\"\n\n    def __str__(self) -> str:\n        return (\n            f\"[{self.name}]\"\n            if self.parameters is None\n            else f\"[{self.name}={self.parameters}]\"\n        )\n\n\ndef _parse(markup: str) -> Iterable[Tuple[int, Optional[str], Optional[Tag]]]:\n    \"\"\"Parse markup in to an iterable of tuples of (position, text, tag).\n\n    Args:\n        markup (str): A string containing console markup\n\n    \"\"\"\n    position = 0\n    _divmod = divmod\n    _Tag = Tag\n    for match in RE_TAGS.finditer(markup):\n        full_text, escapes, tag_text = match.groups()\n        start, end = match.span()\n        if start > position:\n            yield start, markup[position:start], None\n        if escapes:\n            backslashes, escaped = _divmod(len(escapes), 2)\n            if backslashes:\n                # Literal backslashes\n                yield start, \"\\\\\" * backslashes, None\n                start += backslashes * 2\n            if escaped:\n                # Escape of tag\n                yield start, full_text[len(escapes) :], None\n                position = end\n                continue\n        text, equals, parameters = tag_text.partition(\"=\")\n        yield start, None, _Tag(text, parameters if equals else None)\n        position = end\n    if position < len(markup):\n        yield position, markup[position:], None\n",
 "real_bugs/tests/conftest.py": "\"\"\"Makes the parent real_bugs/ directory importable so tests can import the bug modules.\"\"\"\nimport os\nimport sys\n\nsys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), \"..\")))\n",
 "real_bugs/tests/test_real_01_requests_content_type.py": "import real_01_requests_content_type as mod\n\n\ndef test_malformed_param_ignored():\n    content_type, params = mod._parse_content_type_header(\n        \"multipart/form-data; boundary=something; no_equals\"\n    )\n    assert content_type == \"multipart/form-data\"\n    assert params == {\"boundary\": \"something\"}\n\n\ndef test_normal_params():\n    content_type, params = mod._parse_content_type_header(\"text/html; charset=utf-8\")\n    assert content_type == \"text/html\"\n    assert params == {\"charset\": \"utf-8\"}\n\n\ndef test_multiple_malformed_params():\n    content_type, params = mod._parse_content_type_header(\n        \"application/json; foo; bar=baz; qux\"\n    )\n    assert content_type == \"application/json\"\n    assert params == {\"bar\": \"baz\"}\n",
 "real_bugs/tests/test_real_02_requests_super_len.py": "import real_02_requests_super_len as mod\n\n\ndef test_ascii_str():\n    assert mod.super_len(\"hello\") == 5\n\n\ndef test_multibyte_str_counts_bytes():\n    assert mod.super_len(\"café\") == 5\n\n\ndef test_emoji_counts_bytes():\n    assert mod.super_len(\"😀\") == 4\n",
 "real_bugs/tests/test_real_03_requests_netrc_auth.py": "import real_03_requests_netrc_auth as mod\n\n\ndef test_empty_default_credentials_ignored(tmp_path, monkeypatch):\n    netrc_path = tmp_path / \".netrc\"\n    monkeypatch.setenv(\"NETRC\", str(netrc_path))\n    netrc_path.write_text(\"machine example.com login user password pass\\ndefault\\n\")\n    assert mod.get_netrc_auth(\"http://httpbin.org/\") is None\n\n\ndef test_matching_machine_returns_auth(tmp_path, monkeypatch):\n    netrc_path = tmp_path / \".netrc\"\n    monkeypatch.setenv(\"NETRC\", str(netrc_path))\n    netrc_path.write_text(\"machine example.com login user password pass\\n\")\n    assert mod.get_netrc_auth(\"http://example.com/\") == (\"user\", \"pass\")\n\n\ndef test_no_netrc_file_returns_none(tmp_path, monkeypatch):\n    monkeypatch.setenv(\"NETRC\", str(tmp_path / \".netrc\"))\n    assert mod.get_netrc_auth(\"http://example.com/\") is None\n",
 "real_bugs/tests/test_real_04_httpx_primitive_value.py": "import pytest\n\nimport real_04_httpx_primitive_value as mod\n\n\ndef test_valid_primitives():\n    assert mod.primitive_value_to_str(\"a\") == \"a\"\n    assert mod.primitive_value_to_str(1) == \"1\"\n    assert mod.primitive_value_to_str(1.5) == \"1.5\"\n    assert mod.primitive_value_to_str(True) == \"true\"\n    assert mod.primitive_value_to_str(False) == \"false\"\n    assert mod.primitive_value_to_str(None) == \"\"\n\n\ndef test_bytes_raises():\n    with pytest.raises(\n        TypeError, match=\"Expected str, int, float, bool, or None. Got 'bytes'.\"\n    ):\n        mod.primitive_value_to_str(b\"bytes\")\n\n\ndef test_list_and_dict_raise():\n    with pytest.raises(TypeError):\n        mod.primitive_value_to_str([1, 2])\n    with pytest.raises(TypeError):\n        mod.primitive_value_to_str({\"a\": 1})\n",
 "real_bugs/tests/test_real_05_httpx_text_chunker.py": "import real_05_httpx_text_chunker as mod\n\n\ndef test_empty_content_yields_nothing():\n    assert mod.TextChunker().decode(\"\") == []\n\n\ndef test_plain_content():\n    assert mod.TextChunker().decode(\"hello\") == [\"hello\"]\n\n\ndef test_chunked_content():\n    chunker = mod.TextChunker(chunk_size=5)\n    assert chunker.decode(\"hello world\") == [\"hello\", \" worl\"]\n    assert chunker.flush() == [\"d\"]\n",
 "real_bugs/tests/test_real_06_httpx_no_proxy.py": "import real_06_httpx_no_proxy as mod\n\nPROXY_VARS = [\n    \"http_proxy\",\n    \"https_proxy\",\n    \"all_proxy\",\n    \"no_proxy\",\n    \"HTTP_PROXY\",\n    \"HTTPS_PROXY\",\n    \"ALL_PROXY\",\n    \"NO_PROXY\",\n]\n\n\ndef _set_env(monkeypatch, env):\n    for var in PROXY_VARS:\n        monkeypatch.delenv(var, raising=False)\n    for key, value in env.items():\n        monkeypatch.setenv(key, value)\n\n\ndef test_no_proxy_ipv4(monkeypatch):\n    _set_env(monkeypatch, {\"no_proxy\": \"127.0.0.1\"})\n    assert mod.get_environment_proxies() == {\"all://127.0.0.1\": None}\n\n\ndef test_no_proxy_ipv6_and_localhost(monkeypatch):\n    _set_env(monkeypatch, {\"no_proxy\": \"::1\"})\n    assert mod.get_environment_proxies() == {\"all://[::1]\": None}\n    _set_env(monkeypatch, {\"no_proxy\": \"localhost\"})\n    assert mod.get_environment_proxies() == {\"all://localhost\": None}\n\n\ndef test_no_proxy_domain_suffix(monkeypatch):\n    _set_env(monkeypatch, {\"no_proxy\": \"github.com\"})\n    assert mod.get_environment_proxies() == {\"all://*github.com\": None}\n",
 "real_bugs/tests/test_real_07_httpx_urlencode_space.py": "import real_07_httpx_urlencode_space as mod\n\n\ndef test_spaces_encoded_as_percent_20():\n    assert mod.build_query_string([(\"q\", \"hello world\")]) == \"q=hello%20world\"\n\n\ndef test_slash_left_alone():\n    assert mod.build_query_string([(\"a\", \"x/y\")]) == \"a=x/y\"\n\n\ndef test_plain_pairs():\n    assert mod.build_query_string([(\"a\", \"1\"), (\"b\", \"2\")]) == \"a=1&b=2\"\n",
 "real_bugs/tests/test_real_08_httpx_ipv4_regex.py": "import pytest\n\nimport real_08_httpx_ipv4_regex as mod\n\n\ndef test_ipv4_like_hostname_not_rejected():\n    assert mod.encode_host(\"023b76x43144\") == \"023b76x43144\"\n\n\ndef test_valid_ipv4():\n    assert mod.encode_host(\"1.2.3.4\") == \"1.2.3.4\"\n\n\ndef test_invalid_ipv4_and_plain_hostname():\n    with pytest.raises(mod.InvalidURL):\n        mod.encode_host(\"256.1.1.1\")\n    assert mod.encode_host(\"example.com\") == \"example.com\"\n",
 "real_bugs/tests/test_real_09_httpx_gen_delims.py": "import real_09_httpx_gen_delims as mod\n\nPATH = \"/!$&'()*+,;= abc ABC 123 :/[]@\"\n\n\ndef test_path_percent_encoding():\n    path, query, fragment = mod.encode_url_parts(PATH, None, None)\n    assert path == \"/!$&'()*+,;=%20abc%20ABC%20123%20:/[]@\"\n\n\ndef test_query_percent_encoding():\n    path, query, fragment = mod.encode_url_parts(\"/\", PATH + \"?\", None)\n    assert query == \"/!$&'()*+,;=%20abc%20ABC%20123%20:/[]@?\"\n\n\ndef test_fragment_percent_encoding():\n    path, query, fragment = mod.encode_url_parts(\"/\", None, PATH + \"?#\")\n    assert fragment == \"/!$&'()*+,;=%20abc%20ABC%20123%20:/[]@?#\"\n",
 "real_bugs/tests/test_real_10_rich_escape.py": "import real_10_rich_escape as mod\n\n\ndef test_at_tag_escaped():\n    assert mod.escape(\"[@click]Hello[/@click]\") == \"\\\\[@click]Hello\\\\[/@click]\"\n\n\ndef test_bold_tag_escaped():\n    assert mod.escape(\"[bold]hi[/bold]\") == \"\\\\[bold]hi\\\\[/bold]\"\n\n\ndef test_plain_text_unchanged():\n    assert mod.escape(\"plain text\") == \"plain text\"\n",
 "real_bugs/tests/test_real_11_rich_markup_parse.py": "import real_11_rich_markup_parse as mod\n\n\ndef test_nested_bracket_not_one_tag():\n    result = list(mod._parse(\"[foo bar [baz] qux]\"))\n    assert result == [\n        (9, \"[foo bar \", None),\n        (9, None, mod.Tag(\"baz\", None)),\n        (14, \" qux]\", None),\n    ]\n\n\ndef test_simple_tags():\n    result = list(mod._parse(\"[bold]hi[/bold]\"))\n    assert result == [\n        (0, None, mod.Tag(\"bold\", None)),\n        (8, \"hi\", None),\n        (8, None, mod.Tag(\"/bold\", None)),\n    ]\n\n\ndef test_no_tags():\n    assert list(mod._parse(\"no tags\")) == [(0, \"no tags\", None)]\n"
}

for rel, content in FILES.items():
    path = Path("/kaggle/working") / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)

sys.path.insert(0, "/kaggle/working")
import agent

# Smoke test: the harness works and the first bug really fails before any fix.
passed, out = agent.run_tests(
    Path("/kaggle/working/bugs/tests/test_bug_01_off_by_one.py"),
    cwd=Path("/kaggle/working/bugs"),
)
print("smoke test, bug_01 original:", "PASS (unexpected!)" if passed else "FAIL (expected)")


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# QAT 4-bit E4B — the only permitted submission model (bf16 is NOT allowed).
MODEL_ID = "google/gemma-4-e4b-it-qat-w4a16-ct"

model_path = None
# 1. Attached model input (the L4x4 path — L4 has no internet, so the
# model must be attached: right panel → Input → Add input → Models).
from pathlib import Path as _P
if _P("/kaggle/input").exists():
    for _d in sorted(_P("/kaggle/input").glob("**/config.json")):
        if "gemma" in str(_d).lower():
            model_path = str(_d.parent)
            print("using attached model input:", model_path)
            break
# 2. Kaggle download (only if the session has internet; L4 does not).
if model_path is None:
    try:
        import kagglehub
        # Confirmed 26 Nov from the Kaggle model page: the QAT checkpoint
        # is under the "Other" framework, variant gemma-4-e4b-it-qat-w4a16-ct.
        model_path = kagglehub.model_download(
            "google/gemma-4/Other/gemma-4-e4b-it-qat-w4a16-ct"
        )
        print("downloaded via kagglehub:", model_path)
    except Exception as e:
        print("kagglehub failed, falling back to Hugging Face:", e)

load_from = model_path or MODEL_ID
tokenizer = AutoTokenizer.from_pretrained(load_from)
model = AutoModelForCausalLM.from_pretrained(
    load_from,
    # Do NOT force bf16 here: it dequantizes the QAT 4-bit weights to
    # ~18 GB and the generate call OOMs on a T4. Let compressed-tensors
    # load the checkpoint in native 4-bit.
    # L4x4 only: the whole 13.85 GB model fits on ONE 24 GB L4, so load
    # it whole on GPU 0 with a single-device map. device_map="auto" is
    # NOT used: on a 4-GPU node auto still runs accelerate dispatch and
    # the top-level Gemma4ForConditionalGeneration ends up with a
    # wrapped forward but no _hf_hook, so generate() raises
    # AttributeError (28 Nov 01:38 IST run). A single-device map
    # attaches no hooks at all, which is what fixes it. Run this
    # notebook on GPU L4 x4 with the model attached (L4 has no
    # internet, so the attached input is required).
    # On T4x2 this OOMs — the split path kept hitting accelerate edge
    # cases (balanced not splitting, load OOM during quantization,
    # dispatch not attaching the top-level _hf_hook, layer_scalar
    # dropped from the map), so the split is abandoned.
    device_map={"": 0},
    # MUST stay eager: Gemma4AudioModel (part of the full multimodal
    # Gemma4ForConditionalGeneration checkpoint) does not support SDPA —
    # transformers raises ValueError and tells you to use eager instead
    # (huggingface/transformers#28005). SDPA was the 23:21 crash.
    attn_implementation="eager",
    low_cpu_mem_usage=True,
)
model.eval()
# OOM diagnosis (26 Nov): if the first param is bf16/fp32 instead of
# a 4-bit quantized weight, the checkpoint was dequantized on load and
# will OOM on a single T4. Print it so the run log shows it.
first = next(model.parameters())
total = sum(p.numel() for p in model.parameters())
print("model loaded:", load_from)
print(f"first param dtype: {first.dtype}, total params: {total / 1e9:.2f}B")
print("quantization_config:", getattr(model.config, "quantization_config", None))
# Dispatch check (28 Nov _hf_hook fix): with a single-device map
# accelerate attaches NO hooks, so the top-level module must NOT have
# a wrapped forward / _hf_hook. If "has _hf_hook" prints True, STOP —
# do not call generate(). Expected: False, {'': 0}, cuda:0.
print("has _hf_hook:", hasattr(model, "_hf_hook"))
print("hf_device_map:", getattr(model, "hf_device_map", None))
print("model.device:", model.device)
for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i} after load: "
        f"allocated {torch.cuda.memory_allocated(i) / 1e9:.1f} GB, "
        f"reserved {torch.cuda.memory_reserved(i) / 1e9:.1f} GB"
    )


In [ ]:
def generate(prompt: str, greedy: bool = False, max_new_tokens: int = 512) -> str:
    """Gemma-backed generate() for agent.run_all.

    greedy=True: deterministic decoding (do_sample=False) — the first
    candidate of each round, which gives a stable baseline. greedy=False:
    sampled (temperature 0.7, top_p 0.95) — the fallback candidates that
    beat run-to-run sampling variance.
    """
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    torch.cuda.empty_cache()  # free fragmented memory before generate
    gen_kwargs = {"max_new_tokens": max_new_tokens}
    if greedy:
        gen_kwargs["do_sample"] = False
    else:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = 0.7
        gen_kwargs["top_p"] = 0.95
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )


print(generate("Write one line of Python that adds 1 and 2. Reply with code only.", max_new_tokens=64))


In [ ]:
from pathlib import Path

results = agent.run_all(
    generate,
    Path("/kaggle/working/bugs"),
    Path("/kaggle/working/runs"),
    max_attempts=3,
)
print()
print(agent.results_table(results))


In [ ]:
from pathlib import Path

print("## Real bugs")
real_results = agent.run_all(
    generate,
    Path("/kaggle/working/real_bugs"),
    Path("/kaggle/working/runs_real"),
    max_attempts=3,
    pattern="real_*.py",
)
print()
print(agent.results_table(real_results))


In [ ]:
from pathlib import Path

md = (
    "# Baseline results\n\n"
    "Model: google/gemma-4-e4b-it-qat-w4a16-ct (QAT 4-bit, Kaggle L4x4)\n"
    "Date: (fill in)\n\n"
    "## Dev bugs (10 synthetic)\n\n"
    + agent.results_table(results)
    + "\n## Real bugs (11 from requests/httpx/rich)\n\n"
    + agent.results_table(real_results)
)
Path("/kaggle/working/baseline_results.md").write_text(md)
print(md)
print("saved to /kaggle/working/baseline_results.md — copy it back to the repo")


## Next steps

- **More bugs:** add harder ones (wrong loop bounds in real code, type errors,
  broken imports) to `bugs/` and re-run. The loop needs no changes.
- **LoRA fine-tuning:** fine-tune the QAT E4B base on before/after fix
  pairs (the competition allows LoRA; the small-model bonus still
  applies). Start from the fixes this agent produces, corrected by hand.
- **Submission:** the final competition submission is a submission.zip
  with an ADK agent.yaml — this notebook is dev tooling for the loop, not
  the submission. Deadline Thu 3 Dec 2026, 05:29 IST.